# 03 - Create lineage-aware splits and a FASTQ smoke-test plan

This notebook assigns entire clonal complexes to train, development, or test partitions. It then selects a small, phenotype-stratified acquisition cohort for the first sequence-download and assembly smoke test.

## 1. Configuration

The smoke cohort is intentionally small. Begin with three isolates when invoking the downloader, inspect the files, and only then retrieve the full smoke plan.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from disinfectant_tolerance.splits import SplitFractions, assign_grouped_splits

CONFIG_PATH = PROJECT_ROOT / "configs" / "feasibility.json"
RESOLVED_MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "isolate_manifest_resolved.csv"
RETRIEVAL_PLAN_PATH = PROJECT_ROOT / "data" / "manifests" / "ena_run_retrieval_plan.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests"
SMOKE_ISOLATES = 12

config = json.loads(CONFIG_PATH.read_text())
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

## 2. Assign lineage-aware splits

All members of one clonal complex remain in the same partition. This is stricter and more informative than a random isolate split.

In [ ]:
manifest = pd.read_csv(RESOLVED_MANIFEST_PATH, dtype=str).fillna("")
rows = manifest.to_dict(orient="records")
assignments = assign_grouped_splits(
    rows,
    fractions=SplitFractions(**config["split_fractions"]),
    seed=config["random_seed"],
)
manifest["split"] = manifest["group"].map(assignments)

split_path = PROCESSED_DIR / "isolate_splits.csv"
manifest.to_csv(split_path, index=False)

summary = pd.crosstab(manifest["split"], manifest["label"], margins=True)
print(summary)
print("\nGroups crossing partitions:", manifest.groupby("group")["split"].nunique().gt(1).sum())
print("Wrote:", split_path.relative_to(PROJECT_ROOT))

## 3. Select a small acquisition cohort

The smoke test checks network retrieval, paired-end file handling, and the future assembly workflow. It is not used to report model performance.

In [ ]:
retrieval_plan = pd.read_csv(RETRIEVAL_PLAN_PATH, dtype={"label": str}).fillna("")
eligible = manifest[
    (manifest["resolution_status"] == "resolved")
    & manifest["isolate_id"].isin(retrieval_plan["isolate_id"])
].copy()

def sample_stratified(frame: pd.DataFrame, size: int, seed: int) -> pd.DataFrame:
    counts = frame["label"].value_counts().sort_index()
    selected_parts = []
    for label, count in counts.items():
        target = max(1, round(size * count / len(frame)))
        selected_parts.append(frame[frame["label"] == label].sample(n=min(target, count), random_state=seed))
    selected = pd.concat(selected_parts).drop_duplicates("isolate_id")
    if len(selected) > size:
        selected = selected.sample(n=size, random_state=seed)
    elif len(selected) < size:
        remaining = frame[~frame["isolate_id"].isin(selected["isolate_id"])]
        selected = pd.concat([selected, remaining.sample(n=size - len(selected), random_state=seed)])
    return selected

smoke_isolates = sample_stratified(eligible, min(SMOKE_ISOLATES, len(eligible)), config["random_seed"])
smoke_plan = retrieval_plan[retrieval_plan["isolate_id"].isin(smoke_isolates["isolate_id"])].copy()
smoke_plan = smoke_plan.merge(smoke_isolates[["isolate_id", "split"]], on="isolate_id", how="left")
smoke_plan = smoke_plan.sort_values(["label", "isolate_id", "run_accession"]).reset_index(drop=True)

smoke_path = MANIFEST_DIR / "ena_smoke_retrieval_plan.csv"
smoke_plan.to_csv(smoke_path, index=False)

print("Smoke isolates:", smoke_plan["isolate_id"].nunique())
print("Smoke runs:", len(smoke_plan))
print("Estimated FASTQ download: %.2f GB" % (pd.to_numeric(smoke_plan["fastq_bytes"]).sum() / 1_000_000_000))
print("\nSmoke labels:")
print(smoke_isolates["label"].value_counts().sort_index())
print("\nWrote:", smoke_path.relative_to(PROJECT_ROOT))
smoke_plan[["isolate_id", "label", "group", "split", "run_accession", "fastq_bytes"]]

## 4. Download only three isolates first

Run this command from the project root after reviewing the plan:

```bash
python scripts/download_fastq_plan.py \
  --plan data/manifests/ena_smoke_retrieval_plan.csv \
  --output-dir data/raw/fastq \
  --limit-isolates 3
```

The next notebook should assemble and quality-check this small subset before expanding the download.